<a href="https://colab.research.google.com/github/JillianneKateLBocol/Final-Project-CS2/blob/main/Kamia_Ecommerce_NB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
with open("/content/ecommerce.json") as f:
  orders = json.load(f)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
!pip install firebase-admin

In [ ]:
import firebase_admin
from firebase_admin import credentials

path_to_key = "/content/firebase.json"

if not firebase_admin._apps:
    cred = credentials.Certificate(path_to_key)
    firebase_admin.initialize_app(cred, {
        'databaseURL': "https://kamia-ecommerce-db-default-rtdb.asia-southeast1.firebasedatabase.app/"
    })
    print("Firebase connected successfully!")
else:
    print("Firebase is already connected.")

Firebase is already connected.


In [ ]:
from firebase_admin import db

ref = db.reference("orders")

for order in orders:
    ref.child(str(order["order_id"])).set(order)

print("Data uploaded successfully!")

Data uploaded successfully!


In [ ]:
ref = db.reference("orders")
orders_data = ref.get()

In [ ]:
while True:
    print("\n===== ECOMMERCE DATABASE MENU =====")
    print("1. Display All Orders")
    print("2. Add New Order")
    print("3. Update Order")
    print("4. Delete Order")
    print("5. Features")
    print("6. Exit")

    choice = input("Enter choice: ")

    if not choice:
        print("Error: choice required")
        continue

    if not choice.isdigit():
        print("Error: Input number")
        continue

    # 1. Display
    if choice == "1":
        orders_data = ref.get()
        if orders_data:
            print("\n" + "="*50)
            print("            DETAILED ORDER HISTORY")
            print("="*50)

            items_to_show = orders_data.items() if isinstance(orders_data, dict) else enumerate(orders_data)

            for oid, data in items_to_show:
                if not data: continue
                print(f"\n[ ORDER ID: {oid} ]")
                print(f"Customer: {data.get('customer', 'N/A')}")
                print(f"Date:     {data.get('order_date', 'N/A')}")
                print(f"Status:   {data.get('status', 'N/A')}")
                print(f"Payment:  {data.get('payment_method', 'N/A')}")
                print(f"Address:  {data.get('shipping_address', 'N/A')}")

                print("-" * 30)
                print("ITEMS PURCHASED:")
                current_items = data.get('items', [])
                if not current_items:
                    print("  (No items listed)")
                else:
                    for i, item in enumerate(current_items, 1):
                        p_name = item.get('product', 'Unknown')
                        p_qty  = item.get('quantity', 0)
                        p_price = item.get('price', 0)
                        p_cat   = item.get('category', 'N/A')
                        print(f"  {i}. {p_name} [{p_cat}]")
                        print(f"     Qty: {p_qty} | Price: ₱{p_price} | Subtotal: ₱{p_qty * p_price}")

                print("-" * 30)
                print(f"GRAND TOTAL: ₱{data.get('total_amount', 0)}")
                print("="*50)
        else:
            print("\nNo orders found in the database.")

    # 2. Add
    elif choice == "2":
        while True:
            oid = input("Enter Order ID: ")
            if not oid: print("Error: Order ID required"); continue
            if not oid.isdigit(): print("Error: Input number"); continue
            break

        while True:
            customer = input("Enter Customer Name: ")
            if not customer: print("Error: Customer required"); continue
            break

        while True:
            order_date = input("Enter Order Date (YYYY-MM-DD): ")
            if not order_date: print("Error: Order date required"); continue
            if any(c.isalpha() for c in order_date): print("Error: Input number"); continue
            break

        while True:
            payment_method = input("Enter Payment Method: ")
            if not payment_method: print("Error: Payment method required"); continue
            break

        while True:
            shipping_address = input("Enter Shipping Address: ")
            if not shipping_address: print("Error: Shipping address required"); continue
            break

        status = "Pending"
        items = []

        print("\nAdd Items to Order")
        while True:
            product = input("Enter Product Name (or 'done' to finish): ")
            if product.lower() == 'done':
                if not items:
                    print("Error: Product required (Add at least one item)"); continue
                break
            if not product: print("Error: Product required"); continue

            while True:
                category = input("Enter Category: ")
                if not category: print("Error: Category required"); continue
                break

            while True:
                price_in = input("Enter Price: ")
                if not price_in: print("Error: Price required"); continue
                try:
                    price = float(price_in)
                    break
                except ValueError: print("Error: Input number")

            while True:
                qty_in = input("Enter Quantity: ")
                if not qty_in: print("Error: Quantity required"); continue
                try:
                    quantity = int(qty_in)
                    break
                except ValueError: print("Error: Input number")

            items.append({
                "category": category,
                "price": price,
                "product": product,
                "quantity": quantity
            })
            print(f" Added {product}")

        total_amount = sum(item['price'] * item['quantity'] for item in items)
        order_data = {
            "order_id": int(oid),
            "customer": customer,
            "order_date": order_date,
            "status": status,
            "payment_method": payment_method,
            "shipping_address": shipping_address,
            "items": items,
            "total_amount": total_amount
        }

        ref.child(oid).set(order_data)
        print(f"\nOrder #{oid} created successfully!")

    # 3. Update
    elif choice == "3":
        while True:
            oid = input("Enter Order ID to update: ")
            if not oid: print("Error: Order ID required"); continue
            if not oid.isdigit(): print("Error: Input number"); continue
            break

        order_ref = ref.child(oid)
        order_data = order_ref.get()

        if order_data:
            while True:
                new_address = input(f"New Address [{order_data.get('shipping_address')}]: ")
                if not new_address: print("Error: Shipping address required"); continue
                break
            order_ref.update({"shipping_address": new_address})
            print("Update successful!")
        else:
            print("Order ID not found.")

    # 4. Delete
    elif choice == "4":
        oid = input("Enter Order ID to delete: ")
        if ref.child(oid).get():
            ref.child(oid).delete()
            print(f"Order #{oid} deleted.")
        else:
            print("Order ID not found.")

    # 5. Features
    elif choice == "5":
        while True:
            print("\n===== ECOMMERCE DATABASE FEATURES =====")
            print("1. Show total orders and total revenue")
            print("2. Show best-selling product")
            print("3. Show top 3 customers")
            print("4. Show number of orders by status")
            print("5. Search sales of a product")
            print("6. Back to main menu")

            choice_feature = input("Choice: ")

            if not choice_feature:
                print("Error: choice required")
                continue

            orders_data = ref.get()
            if not orders_data: print("No data available."); break

            orders = list(orders_data.values()) if isinstance(orders_data, dict) else orders_data
            match choice_feature:
                case "1":
                    total_orders = len([o for o in orders if o])
                    total_revenue = sum(order["total_amount"] for order in orders if order)
                    print(f"\nTotal Orders: {total_orders}")
                    print(f"Total Revenue: ₱{total_revenue}")
                case "2":
                    product_sales = {}
                    for order in orders:
                        if order and "items" in order:
                            for item in order["items"]:
                                p = item["product"]
                                product_sales[p] = product_sales.get(p, 0) + item["quantity"]
                    if product_sales:
                        best_product = max(product_sales, key=product_sales.get)
                        print(f"\nBest-Selling Product: {best_product}")
                        print(f"Total Quantity Sold: {product_sales[best_product]}")
                case "3":
                    customer_spending = {}
                    for order in orders:
                        if order:
                            c = order["customer"]
                            customer_spending[c] = customer_spending.get(c, 0) + order["total_amount"]
                    top_customers = sorted(customer_spending.items(), key=lambda x: x[1], reverse=True)[:3]
                    print("\nTop 3 Customers:")
                    for i, (customer, amount) in enumerate(top_customers, 1):
                        print(f"{i}. {customer} - ₱{amount}")
                case "4":
                    status_count = {"Pending": 0, "Shipped": 0, "Delivered": 0}
                    for order in orders:
                        if order and order["status"] in status_count:
                            status_count[order["status"]] += 1
                    print("\nOrder Status Count:")
                    for status, count in status_count.items():
                        print(f"{status}: {count}")
                case "5":
                    search_product = input("Enter product name: ").lower()
                    total_qty = 0
                    for order in orders:
                        if order and "items" in order:
                            for item in order["items"]:
                                if item["product"].lower() == search_product:
                                    total_qty += item["quantity"]
                    print(f"\nTotal '{search_product}' sold: {total_qty}")
                case "6":
                    break
                case _:
                    print("Invalid choice.")

    # 6. Exit
    elif choice == "6":
        print("Exiting...")
        break
    else:
        print("Invalid choice.")


===== ECOMMERCE DATABASE MENU =====
1. Display All Orders
2. Add New Order
3. Update Order
4. Delete Order
5. Features
6. Exit
Order ID not found.

===== ECOMMERCE DATABASE MENU =====
1. Display All Orders
2. Add New Order
3. Update Order
4. Delete Order
5. Features
6. Exit
Order #1 deleted.

===== ECOMMERCE DATABASE MENU =====
1. Display All Orders
2. Add New Order
3. Update Order
4. Delete Order
5. Features
6. Exit
Order #2 deleted.

===== ECOMMERCE DATABASE MENU =====
1. Display All Orders
2. Add New Order
3. Update Order
4. Delete Order
5. Features
6. Exit
Order ID not found.

===== ECOMMERCE DATABASE MENU =====
1. Display All Orders
2. Add New Order
3. Update Order
4. Delete Order
5. Features
6. Exit
Order ID not found.

===== ECOMMERCE DATABASE MENU =====
1. Display All Orders
2. Add New Order
3. Update Order
4. Delete Order
5. Features
6. Exit

Add Items to Order
 Added coke
 Added tv

Order #1 created successfully!

===== ECOMMERCE DATABASE MENU =====
1. Display All Orders
2. 